# GANet Evaluation on RWTH-PHOENIX-Weather 2014 (Kaggle)

This notebook runs GANet on the official **RWTH-PHOENIX-Weather 2014** Continuous Sign Language Recognition benchmark on Kaggle GPU.

### Key Differences from ISL-CSLTR:
1. **Sequence Length**: PHOENIX clips are ~100–250 frames long (~5–10s).
2. **Temporal Downsampling**: `num_pool: 2` downsamples by 4x (e.g. 200 frames -> 50 steps), matching the GANet paper architecture.
3. **Vocabulary**: ~1,066 uppercase German weather glosses.
4. **Standard Splits**: Train (5,672), Dev (540), Test (629).

### Step 1: Check GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Unpack GANet Codebase
Ensure `ganet_code.zip` is added via **[+ Add Input]** or uploaded to `/kaggle/input/`.

In [ ]:
import os, glob, shutil

!mkdir -p /kaggle/working/ganet_isl

code_zips = glob.glob('/kaggle/input/**/ganet_code*.zip', recursive=True) + glob.glob('/kaggle/working/**/ganet_code*.zip', recursive=True)
code_dirs = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train_isl.py', recursive=True)]

if code_zips:
    print(f"[*] Unpacking codebase from zip: {code_zips[0]}")
    !unzip -qo {code_zips[0]} -d /kaggle/working/ganet_isl/
elif code_dirs:
    print(f"[*] Found unzipped code in Kaggle input: {code_dirs[0]}")
    !cp -rf {code_dirs[0]}/* /kaggle/working/ganet_isl/
else:
    print("[!] Codebase not found! Upload ganet_code.zip via [+ Add Input].")

%cd /kaggle/working/ganet_isl
!pip install -q opencv-python pyyaml tqdm
print("[✓] Workspace ready at:", os.getcwd())

### Step 3: Locate PHOENIX-2014 Dataset
Add a PHOENIX-2014 dataset to your Kaggle notebook (search in **Add Data** for `phoenix-2014` or `phoenix-weather-2014t`).

In [ ]:
import os, glob

print("=== Available datasets in /kaggle/input/ ===")
!ls -lh /kaggle/input/

# Search for phoenix corpus or annotation files
corpus_files = glob.glob('/kaggle/input/**/*.corpus', recursive=True)
print(f"Found {len(corpus_files)} corpus files:")
for cf in corpus_files[:5]:
    print("  -", cf)

### Step 4: Build PHOENIX Manifest & Verify Feasibility
Generates `data/phoenix_manifest.csv` and verifies CTC alignment with `num_pool=2` (4x downsampling).

In [ ]:
%cd /kaggle/working/ganet_isl

# Build manifest from Kaggle input path
!python tools/make_phoenix_manifest.py \
    --phoenix_root /kaggle/input \
    --manifest_out data/phoenix_manifest.csv \
    --splits_out data/phoenix_splits.json

# Verify CTC feasibility with paper's num_pool = 2 (4x downsampling)
!python tools/verify_data.py \
    --isl_manifest data/phoenix_manifest.csv \
    --splits data/phoenix_splits.json \
    --num_pool 2 \
    --skip_include

### Step 5: Train Baseline GANet on PHOENIX-2014
Matches the GANet paper setup:
- Backbone: ResNet-18 + GCMM + GAM
- Temporal Stage: 2x Conv1d + MaxPool1d (`num_pool: 2`) + 2x BiLSTM + Temporal Attention
- Losses: CTC + Visual GLP (`glp_bins: 12`, `weight: 1.0`)

In [ ]:
%cd /kaggle/working/ganet_isl

!python train_isl.py --config configs/phoenix_ganet.yaml \
    --set work_dir=work_dir/phoenix_ganet_baseline \
          model.num_pool=2 \
          model.vac=false \
          model.glp_tap=visual \
          loss.vac=0.0 \
          loss.kd=0.0 \
          loss.glp=1.0

### Step 6: Evaluate Test Results
Reports final Word Error Rate (WER), Substitutions, Deletions, and Insertions on the standard PHOENIX Test split.

In [ ]:
%cd /kaggle/working/ganet_isl
import json, os

res_path = 'work_dir/phoenix_ganet_baseline/results.json'
if os.path.exists(res_path):
    res = json.load(open(res_path))
    print("=" * 50)
    print("PHOENIX-2014 TEST RESULTS (GANet Baseline)")
    print("=" * 50)
    t = res.get('test', {})
    print(f"Test WER:      {t.get('wer', 0.0):.2f}%")
    print(f"Substitutions: {t.get('sub_rate', 0.0):.2f}%")
    print(f"Deletions:     {t.get('del_rate', 0.0):.2f}%")
    print(f"Insertions:    {t.get('ins_rate', 0.0):.2f}%")
else:
    print(f"Results file not found yet at {res_path}")